Partie 0 – mise en place de l’environnement 
1) Structurer le projet comme suit : 
atelier_tensorflow_iot/ 
│    
├── notebooks/ 
│   
└── atelier_tensorflow_iot.ipynb 
│ 
└── models/ 
└── modele_consommation.keras 
2) Créer le notebook atelier_tensorflow_iot.ipynb 
3) Installer et importer tensorflow,  matplotlib et numpy

In [1]:
%pip install tensorflow matplotlib numpy

  Obtaining dependency information for tensorflow from https://files.pythonhosted.org/packages/8f/a2/6d7e6a738e302530586d484895de2cf3fc158ad9c73b4504a670b2956dd9/tensorflow-2.21.0-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for matplotlib from https://files.pythonhosted.org/packages/bc/be/fa26ed085b41298f64a8f9b7592c671bbf1acc8b0df124c1c5de96b859f8/matplotlib-3.11.1-cp311-cp311-win_amd64.whl.metadata
     ---------------------------------------- 0.0/80.3 kB ? eta -:--:--
     ----- ---------------------------------- 10.2/80.3 kB ? eta -:--:--
     ----- ---------------------------------- 10.2/80.3 kB ? eta -:--:--
     -------------- ----------------------- 30.7/80.3 kB 217.9 kB/s eta 0:00:01
     ------------------- ------------------ 41.0/80.3 kB 245.8 kB/s eta 0:00:01
     -------------------------------------- 80.3/80.3 kB 372.5 kB/s eta 0:00:00
  Obtaining dependency information for numpy from https://files.pythonhosted.org/packages/c5/31/7fc6239c12bce7e9

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

Partie 1 – Génération du dataset 
1) Générer aléatoirement 1000 valeurs pour chacune des variables suivantes : 
a) temperature : valeurs qui suivent une loi normale avec une moyenne de 25 °C et un 
écart-type de 4 °C. 
b) humidite : valeurs réparties de façon uniforme entre 30 % et 80 %. 
c) occupants : valeurs entières choisies entre 1 et 49 inclus. 
2) Déterminer la variable consommation avec la formule suivante : 
 la consommation de base (0 °C, pas d'humidité et pièce vide) est de 50 
 chaque degré supplémentaire augmente la consommation de 5 unités 
 chaque pourcentage d'humidité en plus ajoute 1,5 unité à la consommation 
 chaque personne présente dans la pièce augmente la consommation de 4 unités 
 dans la vraie vie, une formule mathématique parfaite n'existe pas. On ajoute donc une 
petite variation aléatoire (moyenne de 0 et écart-type de 10) pour simuler des imprévus 
ou d'autres facteurs non mesurés. 
3) Rassembler les variables (temperature, humidite et occupants) dans la matrice des 
caractéristiques (features) X de taille 1000x3 en convertissant éventuellement les données au 
format (float32) optimisé pour les calculs 
4)  Créer la cible (target) y qui contiendra la variable consommation, au format float32

1) Générer aléatoirement 1000 valeurs pour chacune des variables suivantes : 

a) temperature : valeurs qui suivent une loi normale avec une moyenne de 25 °C et un 
écart-type de 4 °C. 

b) humidite : valeurs réparties de façon uniforme entre 30 % et 80 %.
 
c) occupants : valeurs entières choisies entre 1 et 49 inclus.

In [3]:
SEED = 42
rng = np.random.default_rng(SEED)
N = 1000


temperature = rng.normal(loc=25, scale=4, size=N)
humidite = rng.uniform(low=30, high=80, size=N)
occupants = rng.integers(low=1, high=50, size=N) 

2) Déterminer la variable consommation avec la formule suivante : 

 la consommation de base (0 °C, pas d'humidité et pièce vide) est de 50 
 chaque degré supplémentaire augmente la consommation de 5 unités 
 chaque pourcentage d'humidité en plus ajoute 1,5 unité à la consommation 
 chaque personne présente dans la pièce augmente la consommation de 4 unités 
 dans la vraie vie, une formule mathématique parfaite n'existe pas. On ajoute donc une 
petite variation aléatoire (moyenne de 0 et écart-type de 10) pour simuler des imprévus 
ou d'autres facteurs non mesurés. 

In [4]:
bruit = rng.normal(loc=0, scale=10, size=N)
consommation = 50 + 5 * temperature + 1.5 * humidite + 4 * occupants + bruit


3) Rassembler les variables (temperature, humidite et occupants) dans la matrice des 
caractéristiques (features) X de taille 1000x3 en convertissant éventuellement les données au 
format (float32) optimisé pour les calculs 

In [5]:
X = np.column_stack([temperature, humidite, occupants]).astype(np.float32)

4)  Créer la cible (target) y qui contiendra la variable consommation, au format float32 

In [6]:
y = consommation.astype(np.float32)

print("X shape :", X.shape, "| dtype :", X.dtype)
print("y shape :", y.shape, "| dtype :", y.dtype)
print("\nAperçu des 5 premières observations :")
for i in range(5):
    print(f"  temp={X[i,0]:5.2f}°C  hum={X[i,1]:5.2f}%  occ={int(X[i,2]):2d}  ->  conso={y[i]:7.2f}")

X shape : (1000, 3) | dtype : float32
y shape : (1000,) | dtype : float32

Aperçu des 5 premières observations :
  temp=26.22°C  hum=62.35%  occ= 5  ->  conso= 285.81
  temp=20.84°C  hum=47.12%  occ=39  ->  conso= 383.05
  temp=28.00°C  hum=50.41%  occ=41  ->  conso= 429.26
  temp=28.76°C  hum=52.00%  occ=47  ->  conso= 458.81
  temp=17.20°C  hum=36.29%  occ=16  ->  conso= 253.27
